# KadiPy Market - Module Aide à la Décision

Ce notebook orchestre tous les autres modules pour fournir des recommandations d'affaires claires (Actionnables) aux coopératives et agriculteurs.

---

## 1. Initialisation de l'Aide à la Décision (`DecisionSupport`)

Le module `DecisionSupport` est le "cerveau" final. Pour fonctionner, il a besoin qu'on lui injecte le module de Prévision (`MarketForecasting`) et le module Logistique (`MarketLogistics`). Il va combiner la géographie et les mathématiques temporelles.

**Pourquoi ?** Un agriculteur n'a pas besoin de savoir ce qu'est un modèle de Fourier ou un Z-Score. Il a besoin qu'on lui dise : *"Vends maintenant"* ou *"Stocke pendant 2 mois"*.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

from kadi.market.logistics import MarketLogistics
from kadi.market.forecasting import MarketForecasting
from kadi.market.decision_support import DecisionSupport

logistics = MarketLogistics()
forecaster = MarketForecasting()
advisor = DecisionSupport(forecasting_module=forecaster, logistics_module=logistics)

print("Module DecisionSupport initialisé avec ses dépendances !")

Module DecisionSupport initialisé avec ses dépendances !


---
## 2. Recommandation d'Arbitrage Spatial (`arbitrage_decision`)

L'arbitrage consiste à acheter où c'est moins cher pour revendre où c'est plus cher. Mais il faut soustraire les coûts de transport. La méthode `arbitrage_decision` simule cette opération commerciale de A à Z.

**Explication de la sortie :**
Ici on demande s'il est malin de transporter 20 tonnes de maïs de Parakou à Cotonou.
Le système renvoie :
- `recommandation` : Une consigne claire ("Favorable", "Défavorable", ou "Risqué").
- `gain_net_total_cfa` : Combien d'argent l'opération va générer *après* avoir payé le transport et absorbé les pertes.
- `gain_net_percent` : Le ROI (Retour sur Investissement).

In [2]:
arbitrage = advisor.arbitrage_decision(crop='maize', market_from='Parakou', market_to='Cotonou', qty_tons=20.0)

print(f"Recommandation : {arbitrage['recommandation']}")
print(f"Gain Net Estimé (après transport) : {arbitrage['gain_net_total_cfa']} XOF")
print(f"Rentabilité de l'opération : {arbitrage['gain_net_percent']}% par rapport à l'investissement brut")

Recommandation : TRANSPORTER
Gain Net Estimé (après transport) : 1730072.0 XOF
Rentabilité de l'opération : 32.83% par rapport à l'investissement brut


---
## 3. Stocker ou Vendre maintenant ? (`storage_vs_sell_now`)

C'est la question numéro 1 après la récolte. La méthode demande au module de prévision quel sera le prix dans quelques mois, soustrait les frais de magasinage/stockage (frais de silos, rongeurs, humidité), et compare avec une vente immédiate.

**Explication de la sortie :**
Si le prix futur prévu n'est pas suffisamment haut pour compenser les mois de stockage payant, le système renvoie `VENDRE_MAINTENANT`. S'il y a un grand profit à attendre de la pénurie future, il indique `STOCKER`.
- `marge_nette_cfa` indique le bénéfice net de la décision.

In [3]:
stockage = advisor.storage_vs_sell_now(crop='maize', market='savalou', current_price=200000.0, qty_tons=5.0)

print(f"Action conseillée : {stockage['recommandation_binaire']}")
print(f"Espérance de marge nette totale (gain additionnel si la recommandation est suivie) : {stockage['marge_nette_cfa']} XOF")

Action conseillée : STOCKER
Espérance de marge nette totale (gain additionnel si la recommandation est suivie) : 1713740.0 XOF


---
## 4. Optimisation de Portefeuille (`portfolio_optimization`)

Comment un agriculteur doit-il répartir ses champs ? La méthode `portfolio_optimization` prend en compte les prévisions du marché, mais aussi la Météo (provenant potentiellement de `kadi.weather`).

**Explication de la sortie :**
Si une sécheresse est annoncée, le système réduit la surface allouée aux cultures gourmandes en eau (maïs) au profit de cultures résilientes (niébé, sorgho).
La sortie offre une recommandation stratégique de résilience, suivie de la répartition exacte en hectares de ce qu'il faut planter sur les terres disponibles.

In [4]:
climate_forecast = {'secheresse_anticipee': True, 'precipitations': 'faibles'}
market_forecast = {'maïs': 'hausse', 'niébé': 'stable'}

portfolio = advisor.portfolio_optimization(available_land_ha=10.0, climate_forecast=climate_forecast, market_forecast=market_forecast)

print(f"Stratégie : {portfolio['recommandation']}")
print("\nRépartition conseillée en Hectares (pour minimiser les risques) :")
for culture, surface in portfolio['repartition_hectares'].items():
    print(f"  - {culture.capitalize()}: {surface} ha")

Stratégie : Privilégier les cultures résistantes à la sécheresse.

Répartition conseillée en Hectares (pour minimiser les risques) :
  - Maïs: 3.0 ha
  - Soja: 3.0 ha
  - Niébé: 4.0 ha
